# Chapter 9 — Unsupervised Learning Techniques
Companion notebook to `part1_fundamentals/09_unsupervised_learning.md`. No downloads needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture, BayesianGaussianMixture
from sklearn.metrics import silhouette_score

## K-Means: choosing k via the elbow method and silhouette score

In [ ]:
X, y_true = make_blobs(n_samples=1000, centers=4, cluster_std=0.7, random_state=42)

inertias, silhouettes = [], []
k_range = range(2, 9)
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    km.fit(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(list(k_range), inertias, "bo-"); axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia"); axes[0].set_title("Elbow method")
axes[1].plot(list(k_range), silhouettes, "go-"); axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette score"); axes[1].set_title("Silhouette score")
plt.show()

In [ ]:
best_k = list(k_range)[int(np.argmax(silhouettes))]
kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=42)
y_pred = kmeans.fit_predict(X)

plt.scatter(X[:, 0], X[:, 1], c=y_pred, cmap="tab10", s=10)
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
            marker="X", s=200, c="black")
plt.title(f"K-Means with k={best_k}")
plt.show()

## DBSCAN: density-based clustering (no k needed, handles arbitrary shapes)

In [ ]:
X_moons, _ = make_moons(n_samples=1000, noise=0.05, random_state=42)

dbscan = DBSCAN(eps=0.2, min_samples=5)
labels = dbscan.fit_predict(X_moons)

plt.scatter(X_moons[:, 0], X_moons[:, 1], c=labels, cmap="tab10", s=10)
plt.title(f"DBSCAN on moons ({len(set(labels)) - (1 if -1 in labels else 0)} clusters found, -1 = noise)")
plt.show()

## Gaussian Mixture Models: soft clustering + generation + BIC/AIC model selection

In [ ]:
bics, aics = [], []
n_range = range(1, 8)
for n in n_range:
    gm = GaussianMixture(n_components=n, n_init=10, random_state=42)
    gm.fit(X)
    bics.append(gm.bic(X))
    aics.append(gm.aic(X))

plt.plot(list(n_range), bics, "bo-", label="BIC")
plt.plot(list(n_range), aics, "go-", label="AIC")
plt.xlabel("Number of components"); plt.legend()
plt.title("Choosing the number of GMM components")
plt.show()

best_n = list(n_range)[int(np.argmin(bics))]
print("Best n_components by BIC:", best_n)

In [ ]:
gm = GaussianMixture(n_components=best_n, n_init=10, random_state=42)
gm.fit(X)

# Generate brand-new synthetic instances!
new_samples, new_labels = gm.sample(10)
print("Sampled 10 new points from the learned distribution:\n", new_samples[:3])

# Anomaly detection via low density
densities = gm.score_samples(X)
density_threshold = np.percentile(densities, 4)
anomalies = X[densities < density_threshold]
plt.scatter(X[:, 0], X[:, 1], s=10, label="normal")
plt.scatter(anomalies[:, 0], anomalies[:, 1], color="red", marker="x", s=60, label="flagged anomaly")
plt.legend(); plt.title("GMM-based anomaly detection (lowest-density 4%)")
plt.show()

## ✍️ Practice
1. Above: does the elbow method agree with the silhouette score on the best k?
2. Why does K-Means struggle on two long, thin, crossing ellipse clusters while DBSCAN handles them fine?
3. Debug: GMM with n_components=5 has 2 components with weight ~0.001 — what does this suggest? Try BayesianGaussianMixture(n_components=10) instead.
4. Bonus: use KMeans(n_clusters=8) to compress an image's colors (load any small photo as an (N,3) RGB array).

In [ ]:
# Your code / notes here